# Baseline Exploration: Multi-Source Sensor Fusion

This notebook demonstrates how to load, align, and fuse high-resolution optical (RGB), thermal (LWIR), and 3D heightmap datasets, and pass them into a 5-channel deep learning model.

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import yaml

print("Libraries loaded successfully.")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Load Configurations & Metadata

In [ ]:
# Load model config
config_path = '../configs/model.yaml'
with open(config_path, 'r') as f:
    model_config = yaml.safe_load(f)

print("Model Configuration:")
print(model_config)

# Load sample metadata
metadata_path = '../data/sample_metadata.csv'
df = pd.read_csv(metadata_path)
df.head()

## 2. Multi-Source Alignment (Spatial Homography)

We align the thermal camera and 3D height sensors to match the optical camera coordinate system using OpenCV perspective warping.

In [ ]:
def generate_mock_data(size=(1024, 1024)):
    """Generates simulated optical, thermal, and depth profile maps for test."""
    # Optical: Light grey plate with a dark scratch defect
    optical = np.ones((size[0], size[1], 3), dtype=np.uint8) * 200
    cv2.line(optical, (200, 300), (800, 320), (50, 50, 50), thickness=5) # Scratch
    
    # Thermal: Cooler area suggesting an adhesion void underneath
    thermal = np.ones(size, dtype=np.float32) * 45.0 # 45 degrees Celsius
    cv2.circle(thermal, (500, 600), 100, 35.0, -1) # Void/cold spot
    
    # 3D Depth: Flat surface with a small blister bump (positive height)
    height = np.zeros(size, dtype=np.float32)
    cv2.circle(height, (350, 450), 60, 150.0, -1) # Blister (150 um high)
    
    return optical, thermal, height

optical, thermal, height = generate_mock_data()
print(f"Optical shape: {optical.shape}, Thermal shape: {thermal.shape}, Height shape: {height.shape}")

In [ ]:
# Example plot of separate modalities
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(optical)
axes[0].set_title("RGB (Optical Cam)")
axes[1].imshow(thermal, cmap='hot')
axes[1].set_title("Thermal/LWIR (°C)")
axes[2].imshow(height, cmap='viridis')
axes[2].set_title("3D Height Map (um)")
plt.show()

## 3. Pixel-level Fusion (5-Channel Tensor)

Combining all inputs into a single data tensor for inference.

In [ ]:
# Standardize values
norm_rgb = optical.astype(np.float32) / 255.0
norm_thermal = (thermal - 20.0) / 60.0 # Standardize to [0, 1]
norm_height = height / 500.0           # Max target thickness metric

# Expand dimensions for stack
norm_thermal = np.expand_dims(norm_thermal, axis=2)
norm_height = np.expand_dims(norm_height, axis=2)

# Construct 5-Channel Fused Tensor
fused_tensor = np.concatenate([norm_rgb, norm_thermal, norm_height], axis=2)
print(f"Fused Tensor Shape: {fused_tensor.shape}") # Expected (1024, 1024, 5)

## 4. PyTorch Multi-Source Network

In [ ]:
class CoatingFusionNet(nn.Module):
    def __init__(self, in_channels=5, num_classes=5):
        super(CoatingFusionNet, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        
        # Classifier head
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool(self.relu(self.conv2(x)))
        out = self.fc(x)
        return out

model = CoatingFusionNet()
sample_input = torch.tensor(fused_tensor).permute(2, 0, 1).unsqueeze(0) # Shape: (1, 5, 1024, 1024)
outputs = model(sample_input)
print(f"Model output shape: {outputs.shape} (Batch, Classes)")